# S5.M8-1 Orange 大版流失的真实 SGB 复核

**目的**：方法筛选第四阶段第 2 步（[DR-M5-015](../../../registry/decision_records/DR-M5-015.yaml) · D-6）。

Orange 大版是 17 个任务里唯一没有真实 SGB 证据的数据集。S5.M5 在它的流失任务上用明文代理得到：
- I2 + I3（优化版联邦 − 原版联邦）+0.036；
- 优化版联邦 − 内地单边 +0.041。

本步在同一输入上用真实隐语 SGB 各拟合一次原版联邦与优化版联邦，检验两件事：
- SGB 与同配置代理之差是否在 0.005 以内（DR-M5-012 · D-1 的容差）；
- I2 + I3 在 SGB 上是否仍成立。

**前提**：S5.M7 判定主动方明文直方图与原版逐位一致（6 个任务完整拟合，不同 0 行），所以用明文直方图版本拟合——结果相同，只是更快。

**口径**：与 S5.M2 相同（`method_screen_sgb.evaluate`）。测试行 bootstrap 1,000 次，单次拟合（seed 11），区间不含训练随机性。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m8_orange_large_sgb.yaml"
m8 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m8["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_large_confirm.py", "sgb_cost_screen.py", "method_screen_sgb.py", "method_screen.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m8["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m8["decision_record"], "| step:", m8["step_id"], "|", m8["purpose"])
print("seed:", m8["seed"], "（每个 SGB 拟合一次）| git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
LARGE, M8 = (pathlib.Path(os.environ[m8[k]]) for k in ("large_dir_env", "out_dir_env"))
print("启动指纹（每段运行开始时记录；路径只留文件名）：")
launch = {}
for line in (M8 / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
        print("  sha256:" + digest[:HASH_PREFIX_LEN], pathlib.Path(path).name)
    else:
        print(" ", line)
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m8_orange_large_sgb.yaml sha256:6631b47341b5
registry/decision_records/DR-M5-015.yaml sha256:12778151bfe0
modules/m5_modeling/components/sgb_large_confirm.py sha256:dc8d6cf4e244
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
决策记录: registry/decision_records/DR-M5-015.yaml | step: S5.M8 | Orange 大版流失上原版联邦与优化版联邦的真实 SGB 测试 AUROC、与同配置代理之差、SGB 内部对比
seed: 11 （每个 SGB 拟合一次）| git: 120e404（工作区含本步未提交改动）
启动指纹（每段运行开始时记录；路径只留文件名）：
  == 2026-11-04 14:26:31 F_opt_p ==
  sha256:dc8d6cf4e244 sgb_large_confirm.py
  sha256:6631b47341b5 s5_m8_orange_large_sgb.yaml
  sha256:078b1b1ee1c4 DR-M5-015.yaml
  sha256:cf6088aa95be sgb_cost_screen.py
  sha256:2e54c618305f method_screen_sgb.py
  sha256:05e19e025d0b method_screen.py
  secretflow 1.13.0b0
  == 2026-11-04 16:31:25 F_def ==
  sha256:dc8d6cf4e244 sgb_large_confirm.p

## 1. 输入与配置

- 输入：S5.M5 主分析的私有输入（每方前 300 个编码列，final 视图）。
- 配置：全部取自 S5.M5 的结果 JSON。
  - 原版联邦：隐语默认配置、10 桶、初始分数 0.5。
  - 优化版联邦：S5.M5 选定配置、33 桶、先验初始分数。
- 下一格列出每个方法的深度、棵数与分桶，以及 D-6 的估计秒数与实际拟合秒数。

In [2]:
from modules.m5_modeling.components import sgb_large_confirm as LC

P = LC.MS.Params(cfg)
NAME = m8["task"]
res = json.loads((LARGE / (NAME + ".json")).read_text(encoding="utf-8"))
plan = json.loads((M8 / ("%s__plan.json" % NAME)).read_text(encoding="utf-8"))
summary = json.loads((M8 / ("%s__summary.json" % NAME)).read_text(encoding="utf-8"))
fits = summary["fits"]
sc = LC.MS.Screen(NAME, cfg, PRIV, M8 / "unused")
fv = sc.v["final"]
print("%s：fit 行 %d、测试行 %d（正例 %d），每方 %d 列" % (NAME, len(fv["fit_y"]), len(fv["ev_y"]), int(fv["ev_y"].sum()),
                                               fv["fit_xg"].shape[1]))
size = pd.DataFrame([{"方法": m, "深度": f["depth"], "棵数": f["trees"], "分桶": f["bins"],
                      "估计秒数（D-6）": plan["estimate_seconds"][m], "实际拟合秒数": f["fit_seconds"]}
                     for m, f in fits.items()])
print(size.to_string(index=False))

orange_large_churn：fit 行 40000、测试行 10000（正例 734），每方 300 列
     方法  深度  棵数  分桶  估计秒数（D-6）  实际拟合秒数
F_opt_p   3  94  33     3921.7  6301.2
  F_def   5 100  10     6953.3  4691.6


## 2. 复现

完整拟合每个方法约 1 ~ 2 小时，notebook 不重拟合 SGB。下一格做三件事：
- 用仓库组件重拟合同配置代理（seed 11）；
- 从存档的 SGB 打分重算测试 AUROC；
- 重算 bootstrap 对比。

要求与批量运行的结果相同。

In [3]:
REPRO_TOL = 1e-12
sgb = {m: np.load(M8 / ("%s__%s.npy" % (NAME, m))).astype(float) for m in fits}
proxy = {m: LC.SG.proxy_pred(P, sc, res, m) for m in fits}
pairs = [p for p in LC.PAIRS if set(p) <= set(fits)]
again = LC.SG.evaluate(P, sc, res, sgb, proxy, pairs)
for m in fits:
    assert abs(LC.MS.auc(fv["ev_y"], proxy[m]) - fits[m]["proxy_seed11_auroc"]) <= REPRO_TOL, m
    assert abs(LC.MS.auc(fv["ev_y"], sgb[m]) - fits[m]["sgb_auroc"]) <= REPRO_TOL, m
    a, b = again["methods"][m]["sgb_minus_proxy"], summary["methods"][m]["sgb_minus_proxy"]
    assert all(abs(a[k] - b[k]) <= REPRO_TOL for k in ("diff", "ci_low", "ci_high", "p")), m
for k, v in again["contrasts"].items():
    assert all(abs(v["sgb"][x] - summary["contrasts"][k]["sgb"][x]) <= REPRO_TOL for x in ("diff", "ci_low", "ci_high", "p")), k
print("用仓库组件重拟合同配置代理（seed 11）、从存档打分重算 AUROC 与 bootstrap 对比：与批量运行相同（差 ≤ %g）" % REPRO_TOL)

用仓库组件重拟合同配置代理（seed 11）、从存档打分重算 AUROC 与 bootstrap 对比：与批量运行相同（差 ≤ 1e-12）


## 3. 结果

- **SGB 与同配置代理**：两者之差的配对区间（测试行 bootstrap），以及差是否在 0.005 以内。
- **SGB 内部对比**：优化版联邦 − 原版联邦（I2 + I3），与 S5.M5 的代理（6 seed）对照。
- **只作描述**：联邦 SGB − 内地单边代理。内地单边只有一方，无需密文，没有 SGB 版本。
- 汇总 CSV 写入 `results/sgb_large_v1/`。

In [4]:
OUT = REPO_ROOT / m8["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R, TOL = m8["round_auc"], m8["proxy_tol"]
rows = []
for m, f in fits.items():
    d = summary["methods"][m]["sgb_minus_proxy"]
    rows.append({"dataset": NAME, "method": m, "trees": f["trees"], "depth": f["depth"], "bins": f["bins"],
                 "sgb_auroc": f["sgb_auroc"], "proxy_auroc": f["proxy_seed11_auroc"], "proxy_6seed_auroc": f["proxy_6seed_auroc"],
                 "sgb_minus_proxy": d["diff"], "ci_low": d["ci_low"], "ci_high": d["ci_high"], "within_tolerance": abs(d["diff"]) <= TOL,
                 "estimate_seconds": plan["estimate_seconds"][m], "fit_seconds": f["fit_seconds"]})
vs = pd.DataFrame(rows)
con = []
for k, v in summary["contrasts"].items():
    px = v["proxy_6seed"] or {}
    con.append({"dataset": NAME, "contrast": k, "sgb_diff": v["sgb"]["diff"], "sgb_ci_low": v["sgb"]["ci_low"],
                "sgb_ci_high": v["sgb"]["ci_high"], "sgb_p": v["sgb"]["p"], "proxy_6seed_diff": px.get("diff"),
                "proxy_6seed_ci_low": px.get("ci_low"), "proxy_6seed_ci_high": px.get("ci_high")})
cn = pd.DataFrame(con)
g_t = res["test"]["G_t"]["auroc"]
desc = pd.DataFrame([{"dataset": NAME, "method": m, "sgb_auroc": f["sgb_auroc"], "guest_only_proxy_6seed_auroc": g_t,
                      "sgb_minus_guest_only_proxy": f["sgb_auroc"] - g_t} for m, f in fits.items()])
vs.to_csv(OUT / "sgb_vs_proxy.csv", index=False, float_format="%.5f")
cn.to_csv(OUT / "sgb_contrasts.csv", index=False, float_format="%.5f")
desc.to_csv(OUT / "sgb_vs_guest_only_descriptive.csv", index=False, float_format="%.5f")
print("SGB 与同配置代理（容差 %.3f，测试簇 bootstrap）：" % TOL)
print(vs.round(R).to_string(index=False))
print("SGB 内部对比（与 S5.M5 代理 6 seed 对照）：")
print(cn.round(R).to_string(index=False))
print("只作描述：联邦 SGB − 内地单边代理（6 seed；内地单边无需密文，没有 SGB 版本）：")
print(desc.round(R).to_string(index=False))

SGB 与同配置代理（容差 0.005，测试簇 bootstrap）：
           dataset  method  trees  depth  bins  sgb_auroc  proxy_auroc  proxy_6seed_auroc  sgb_minus_proxy  ci_low  ci_high  within_tolerance  estimate_seconds  fit_seconds
orange_large_churn F_opt_p     94      3    33     0.7526       0.7545             0.7545          -0.0019 -0.0058   0.0023              True            3921.7       6301.2
orange_large_churn   F_def    100      5    10     0.7173       0.7186             0.7186          -0.0013 -0.0162   0.0144              True            6953.3       4691.6
SGB 内部对比（与 S5.M5 代理 6 seed 对照）：
           dataset        contrast  sgb_diff  sgb_ci_low  sgb_ci_high  sgb_p  proxy_6seed_diff  proxy_6seed_ci_low  proxy_6seed_ci_high
orange_large_churn F_opt_p − F_def    0.0353      0.0222       0.0483    0.0            0.0359              0.0229               0.0499
只作描述：联邦 SGB − 内地单边代理（6 seed；内地单边无需密文，没有 SGB 版本）：
           dataset  method  sgb_auroc  guest_only_proxy_6seed_auroc  sgb_minus_guest_only_pr

## 4. 结论

1. **代理在 Orange 大版上可信**：两个方法的 SGB 与同配置代理之差都在 0.005 以内。
   - 优化版联邦：SGB 0.7526，代理 0.7545，差 −0.0019 [−0.0058, +0.0023]；
   - 原版联邦：SGB 0.7173，代理 0.7186，差 −0.0013 [−0.0162, +0.0144]。
   - 与 S5.M2 不同，这里的 SGB 略低于代理，没有出现原版参数被代理低估的情况。
2. **I2 + I3 在真实 SGB 上成立**：优化版联邦 − 原版联邦 = **+0.0353 [+0.0222, +0.0483]**，p < 0.001；S5.M5 的代理（6 seed）为 +0.0359 [+0.0229, +0.0499]，几乎相同。
3. **只作描述**：优化版联邦 SGB 比内地单边代理（6 seed，0.7133）高 0.039，原版联邦 SGB 只高 0.004。与 S5.M5「不做 I2 + I3，联邦看不出增益」一致。
4. **开销**：每方 300 列、明文直方图版本：
   - 优化版联邦 94 棵、深度 3，拟合 6,301 秒（全程与门禁和 notebook 构建争用 CPU，D-6 估算 3,922 秒）；
   - 原版联邦 100 棵、深度 5，拟合 4,692 秒（CPU 空闲，D-6 估算 6,953 秒，启动前试训外推 5,129 秒）。
   - 耗时是单进程两方模拟的合计计算时间，不是部署墙钟。

**对报告的意义**：Orange 大版原本是唯一没有任何真实 SGB 证据的数据集（DR-M5-015 · D-6）。现在流失任务上 I2 + I3 的增益与代理一致地成立，M3 预筛加明文直方图让每方 300 列的单次 SGB 能在 2 小时内完成。

**边界**：
- 每个方法单次拟合（seed 11），区间只含测试行抽样，不含训练随机性；
- 只做了流失一个任务（D-6：三任务中正例最多、增益最大）；追加销售与偏好没有 SGB 复核；
- 明文直方图只在实验代码里替换一个方法，用于正式联调需经安全评审；
- 公开数据上的开发证据，不换算成汇丰的命中率或收益；技术侧自评，未经法务复核（W-001）。